# 01 — Collect data

Sources (see `config.SOURCES`):
- **`opensubtitles_enfi`** — EN-FI parallel pairs from OpenSubtitles (HF, streamed). Dialogue, leans colloquial.
- **`opus_100`** — EN-FI parallel pairs from OPUS-100 (HF, streamed). Broad, mixed-domain.
- **`genius_rap`** — Finnish rap lyrics, FI-only "flavor". Genius blocks lyric scraping (Cloudflare Private Access Tokens), so we use the Genius **API** for song lists only and you paste favourite lines into a **curated seed** (`data/raw/genius_rap/<artist>.txt`).

Config reads `.env` (see `.env.example`): `GENIUS_ACCESS_TOKEN`, `LMSTUDIO_*`.

In [2]:
from puhekieli_llm.config import summary, SOURCES, RAP_ARTISTS
from puhekieli_llm import sources as S
print(summary())
print()
for name, m in SOURCES.items():
    print(f"{name:20s} role={m['role']:7s} register={m['register']:9s} {m['status']}")

task      : en -> fi (puhekieli)
root      : /Users/victormanuel.ontiveros/repos/puhekieli-llm
device    : mps
raw       : /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/raw
clean     : /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/clean
tokenized : /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/tokenized
models    : /Users/victormanuel.ontiveros/repos/puhekieli-llm/models
sources   : opensubtitles_enfi, opus_100, genius_rap, rap_synthetic

opensubtitles_enfi   role=pairs   register=mixed     active
opus_100             role=pairs   register=mixed     active
genius_rap           role=flavor  register=puhekieli active
rap_synthetic        role=synth   register=puhekieli active


## A) Rap lyrics — curated seed (flavor, FI-only)

Genius scraping is blocked, so this is a two-step manual flow:
1. Run the metadata cell to list song **URLs** per artist (uses the working Genius API).
2. Open URLs in your browser, paste lines you like into `data/raw/genius_rap/<artist>.txt` (one line per line; `#` lines tag songs). See `examples/genius_rap_seed.example.txt`.
3. Run the clean cell to build `data/clean/genius_rap.jsonl`.

In [3]:
import os
# Step 1: list songs per artist via the Genius API (metadata only, no lyrics).
if os.getenv('GENIUS_ACCESS_TOKEN'):
    S.fetch_genius_metadata(RAP_ARTISTS, max_songs_per_artist=40)
else:
    print('No GENIUS_ACCESS_TOKEN in .env — skipping song listing.')
    print('You can still paste lyrics manually into data/raw/genius_rap/<artist>.txt')

  Gettomasa: 40 songs listed -> gettomasa.songs.json
  JVG: 40 songs listed -> jvg.songs.json
  Ibe: 16 songs listed -> ibe.songs.json
  Etta: 11 songs listed -> etta.songs.json
  Costi: 27 songs listed -> costi.songs.json

Next: open the URLs, paste lyrics you like into /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/raw/genius_rap/<artist>.txt, then run clean_genius_lyrics().


In [4]:
from puhekieli_llm.config import RAW, CLEAN
# Step 3: clean curated seed .txt files -> data/clean/genius_rap.jsonl
seed_dir = RAW / 'genius_rap'
txts = [f for f in seed_dir.glob('*.txt') if f.stem.lower() not in {'readme','example'}] if seed_dir.exists() else []
if txts:
    S.clean_genius_lyrics()
else:
    print('No seed lyrics yet. Paste lines into data/raw/genius_rap/<artist>.txt then re-run.')

genius_rap: 6016 curated lyric lines -> /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/clean/genius_rap.jsonl


In [5]:
# Peek + puhekieli sanity check on what we collected
from puhekieli_llm.eval import puhekieli_score, corpus_puhekieli_rate
p = CLEAN / 'genius_rap.jsonl'
if p.exists():
    recs = list(S.read_jsonl(p))
    print(f'{len(recs)} lyric lines')
    for r in recs[:8]:
        print(f"  [{puhekieli_score(r['fi'])['score']:+.1f}] {r['fi']}")
    print(f"\nspoken-leaning fraction: {corpus_puhekieli_rate([r['fi'] for r in recs]):.0%}")
else:
    print('genius_rap.jsonl not built yet.')

6016 lyric lines
  [+1.0] Mä voin viedä maailman ympäri sut
  [+1.0] Mun kaa sulla on aina synttärit, boo
  [+1.0] Miks sä mietit niitä? Voidaan lähtee Bogotaan
  [+1.0] Tätä päivää sä et tuu viettämään ny jälleen kotona
  [+1.0] Mä voin viedä maailmal konttorist sut (Ooh-wee)
  [+0.0] Mennää johonki mis on kookoksii puus
  [+1.0] Mä voin viedä maailman ympäri sut (Okay)
  [+1.0] Mun kaa sul on aina synttärit, boo

spoken-leaning fraction: 39%


## B) EN-FI parallel pairs (OpenSubtitles + OPUS-100)

Both stream from HuggingFace — no giant downloads, capped by `MAX_PAIRS`. Keep small while iterating; bump up for real training.

In [6]:
from puhekieli_llm.config import RAW
MAX_PAIRS = 50_000
# OpenSubtitles EN-FI (dialogue). Streamed from HF; safe to run headless.
S.fetch_opensubtitles(max_pairs=MAX_PAIRS)

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

opensubtitles_enfi: 50000 EN-FI pairs -> /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/clean/opensubtitles_enfi.jsonl


PosixPath('/Users/victormanuel.ontiveros/repos/puhekieli-llm/data/clean/opensubtitles_enfi.jsonl')

In [7]:
q = CLEAN / 'opensubtitles_enfi.jsonl'
if q.exists():
    recs = list(S.read_jsonl(q))
    print(f'{len(recs)} EN-FI pairs')
    for r in recs[:6]:
        print(f"  EN: {r['en']}\n  FI: {r['fi']}\n")

50000 EN-FI pairs
  EN: PREVIOUSLY ON "MIND GAMES"...
  FI: Aiemmin...

  EN: THIS IS THE GUY YOU'LL WORK WITH. OKAY. OLIVER.
  FI: - Suunnittelit tätä koko ajan.

  EN: I MEAN, YOUR PARENTS ARE LIKE ZILLIONAIRES OR SOMETHING, RIGHT?
  FI: Miten voit tehdä tämän?

  EN: WHAT DOES THAT MEAN?
  FI: - Ansaitsen ne rahat.

  EN: WHAT IS WITH YOU? I'VE NEVER SEEN YOU LIKE THIS ON A JOB BEFORE.
  FI: Emme suunnittele mitään.

  EN: BECAUSE I'VE NEVER BEEN SCREWING PEOPLE
  FI: Emme yritä saada Milesin rahoja.



In [8]:
# OPUS-100 EN-FI (broad, mixed-domain) — complements the subtitle dialogue.
S.fetch_opus100(max_pairs=MAX_PAIRS)

o = CLEAN / 'opus_100.jsonl'
if o.exists():
    recs = list(S.read_jsonl(o))
    print(f'{len(recs)} EN-FI pairs')
    for r in recs[:4]:
        print(f"  EN: {r['en']}\n  FI: {r['fi']}\n")

opus_100: 50000 EN-FI pairs -> /Users/victormanuel.ontiveros/repos/puhekieli-llm/data/clean/opus_100.jsonl
50000 EN-FI pairs
  EN: Jacob...
  FI: Jacob.

  EN: He's your brother.
  FI: Sinun veljesi.

  EN: - No, please, it was my pleasure.
  FI: Huomaan, että tämä on sinulle tärkeää.

  EN: - How do you do?
  FI: - Hauska tutustua.



## ✅ Next
- `01b_synthesize.ipynb` — back-translate rap lyrics into EN→FI pairs (local LLM).
- then `02_tokenizer.ipynb` — train a BPE tokenizer over subtitles + rap + synth.